# Remote Duckiedrone Docker Contexts

This notebook compares remote Docker context access paths to a physical Duckiedrone.

## Create an authorized Duckiedrone context

A Docker context can use SSH to reach an authorized remote Docker daemon. Creating a context changes only the base station's local Docker configuration; the first command that uses it contacts the remote daemon. Do this only for a Duckiedrone you own or are explicitly authorized by its owner to manage. When another party owns the Duckiedrone or network, follow the device owner's documented access requirements.

```bash
docker context create duckiedrone-DUCKIEDRONE_NAME \
  --description "Authorized Duckiedrone connection" \
  --docker "host=ssh://duckie@DUCKIEDRONE_NAME.local"
docker context inspect duckiedrone-DUCKIEDRONE_NAME
docker --context duckiedrone-DUCKIEDRONE_NAME version
docker context show
```

The explicit `version` command is a read-only connectivity test. It uses the context named with `--context`, but it does not select that remote context for later unqualified commands; `docker context show` continues to report the unqualified selection. A successful result confirms that the chosen context can reach a Docker daemon; it does not authorize changes to that daemon. If the connection fails or times out, stop. Verify the device name and access details through its provisioning record, its owner, or the applicable documented access policy. Do not substitute an unencrypted TCP endpoint, guess port `2375` or `2376`, change `DOCKER_HOST`, or retry against an unfamiliar Duckiedrone.

When an authorized context is no longer needed, remove only its local configuration after confirming that it is not active:

```bash
docker context rm duckiedrone-DUCKIEDRONE_NAME
```

The Duckietown Shell can manage an authorized remote Duckiedrone through its own documented deployment workflow. `dts stack` uses its own subcommands and names a stack as well as the machine; do not assume those commands use the Docker context created above. It is a separate tool and workflow.

## Run the same command two ways

After authorization, an SSH shell and an explicit base-station context can both reach the Docker daemon on the same physical Duckiedrone. The difference is where the Docker client runs: SSH gives you a shell on the Duckiedrone, while `--context` keeps the client in the base-station terminal and directs that one command to the Duckiedrone daemon.

First, use SSH to open a shell on the Duckiedrone:

```bash
ssh duckie@DUCKIEDRONE_NAME.local
```

At the Duckiedrone shell prompt, run read-only Docker commands and then exit:

```bash
docker image ls
docker container ls --all
exit
```

Alternatively, remain on the base station and run the same commands through the authorized context:

```bash
docker --context duckiedrone-DUCKIEDRONE_NAME image ls
docker --context duckiedrone-DUCKIEDRONE_NAME container ls --all
```

Both paths query the same Duckiedrone image cache and container inventory when the context points to that Duckiedrone. The outputs can differ in timing or client formatting, but they describe resources stored by the same remote daemon. An explicit context avoids opening an interactive remote shell; it does not avoid connecting to the Duckiedrone or the need for valid credentials and authorization.

This is useful for an authorized one-command status check, a script that names its intended target, or comparing a local build with the images available on a device. It is not a security boundary. A modifying Docker command uses the same target selection, so do not use either path to experiment with platform containers.

## Compare local and Duckiedrone resources

The base station and a physical Duckiedrone have separate Docker inventories. From the base station, assign `LOCAL_CONTEXT` to the authorized local context and make the comparison explicit:

```bash
LOCAL_CONTEXT=default  # Replace default with your authorized local context name.
docker --context "$LOCAL_CONTEXT" image ls
docker --context "$LOCAL_CONTEXT" container ls --all
docker --context duckiedrone-DUCKIEDRONE_NAME image ls
docker --context duckiedrone-DUCKIEDRONE_NAME container ls --all
```

The local commands show the base station's image cache and learner-owned `lx-docker-*` containers. The authorized remote commands show images and containers stored on that Duckiedrone, including its platform services. An image or container name can exist on both hosts without being the same resource, because each daemon owns a separate inventory. The commands inspect only; do not copy values from an authorized Duckiedrone into reports or use this comparison as a reason to remove or modify platform resources.

## Further reading

See Docker's official guide to [contexts](https://docs.docker.com/engine/manage-resources/contexts/) and the [Duckiedrone DD24 manual](https://docs.duckietown.com/ente/opmanual-dd24/) for physical Duckiedrone management.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
